# RSNA Knee Abnormality Detection — Phase 3 real-data audit

Internet-off Kaggle notebook. **Do not enable internet. Do not attach a GPU.**
This notebook does **not** train a model and does **not** report smoke-test AUC.

| Path | Role |
| --- | --- |
| `/kaggle/input/rsna-knee-abnormality-detection` or `/kaggle/input/competitions/rsna-knee-abnormality-detection` | Competition data (Kaggle UI attach; no API token) |
| Private source dataset | Allowlisted repository ZIP or unpacked `knee/` tree |
| `/kaggle/working/rsna_knee_src` | Extracted working source (the only writable source copy) |
| `/kaggle/working/rsna_audit_outputs` | All generated reports and tables |

`/kaggle/input` is read-only. Radiology-report **text** is never printed or exported.
Three audit levels are labelled `full_metadata`, `one_header_per_study`, and
`sampled_dicom`. Sampled DICOM findings are **not** full-dataset statistics.
Patient-grouped folds are **not** created until PatientID is verified on real data.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_audit_outputs"
MARKERS = ("knee/config.py", "knee/kaggle_audit.py")
SOURCE_SEARCH_MAX_DEPTH = 6
SOURCE_WALK_SKIP_DIRS = frozenset(
    {"train_series", "test_series", ".git", "__pycache__", ".venv", "data"}
)


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _debug_input_layout() -> None:
    print(f"kaggle_input_exists={KAGGLE_INPUT.is_dir()}")
    if not KAGGLE_INPUT.is_dir():
        return
    children = sorted(KAGGLE_INPUT.iterdir())
    print("kaggle_input_top_level=" + ",".join(child.name for child in children))
    for child in children:
        kind = "dir" if child.is_dir() else "file"
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            print(f"input_dataset {child.name} kind={kind} role=competition")
            continue
        extra = ""
        if child.is_dir():
            try:
                names = sorted(p.name for p in child.iterdir())[:30]
            except OSError as exc:
                names = [f"oserror:{exc}"]
            extra = " children=" + ",".join(names)
        print(f"input_dataset {child.name} kind={kind}{extra}")


def _find_source_in_tree(root: Path, max_depth: int = SOURCE_SEARCH_MAX_DEPTH) -> Path | None:
    """Find knee/config.py under a Kaggle input dataset, including GitHub-archive nesting."""
    if _looks_like_source(root):
        return root
    if max_depth <= 0 or not root.is_dir():
        return None
    try:
        root_resolved = root.resolve()
    except OSError:
        return None
    for dirpath, dirnames, _filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_resolved)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(d for d in dirnames if d not in SOURCE_WALK_SKIP_DIRS)
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            return current
    return None


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md"):
        src = src_root / filename
        if src.is_file():
            shutil.copy2(src, dest / filename)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


def locate_and_extract_source() -> Path | None:
    _debug_input_layout()
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips = []
    dirs = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found = _find_source_in_tree(child, SOURCE_SEARCH_MAX_DEPTH)
            if found is not None:
                dirs.append(found)
            zips.extend(
                zip_path for zip_path in sorted(child.glob("*.zip")) if _zip_has_source(zip_path)
            )
    if zips:
        _safe_extract(zips[0], SRC_DST)
        found = _find_source_in_tree(SRC_DST, SOURCE_SEARCH_MAX_DEPTH)
        if found is not None:
            return found
    if dirs:
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
if SOURCE_ROOT is not None:
    resolved = str(SOURCE_ROOT.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"source_root={SOURCE_ROOT}")
else:
    print("source_root=not_found (will rely on an already importable knee package)")

try:
    from knee.kaggle_audit import run_notebook_audit  # noqa: E402
    from knee.kaggle_runtime import detect_kaggle_paths, preflight  # noqa: E402
except ImportError as exc:
    raise SystemExit(
        "AUDIT STOPPED: could not import knee. Attach the private source dataset "
        f"(zip or unpacked tree with knee/config.py). Original error: {exc}"
    ) from exc

print("internet_used=False gpu_required=False training_run=False")
print(detect_kaggle_paths())


In [ ]:
# Preflight: show detected paths, then stop clearly if inputs are missing.
# Do not recurse into the DICOM tree.
from pathlib import Path

from knee.audit import STATUS_DATA_NOT_AVAILABLE
from knee.kaggle_audit import run_notebook_audit
from knee.kaggle_runtime import KaggleLayout, preflight

SAMPLE_SIZE = int(os.environ.get("RSNA_AUDIT_SAMPLE_SIZE", "24"))
layout = KaggleLayout()
# Local / fixture runs may override these; on Kaggle the defaults are correct.
DATA_ROOT = Path(os.environ["RSNA_DATA_ROOT"]) if os.environ.get("RSNA_DATA_ROOT") else None
OUT_DIR = Path(os.environ["RSNA_AUDIT_OUTPUT_DIR"]) if os.environ.get("RSNA_AUDIT_OUTPUT_DIR") else layout.output_dir

pre = preflight(layout=layout, data_root=DATA_ROOT, output_dir=OUT_DIR)
print(pre.message)
print("Detected paths:")
for key, value in pre.to_dict()["layout"].items():
    print(f"  {key}: {value}")
print("Kaggle input datasets (top-level only):", pre.top_level_input_datasets)
print("Competition top-level (not a DICOM walk):", pre.top_level_competition)
print("documented_files_present:", pre.documented_files_present)
print("data_source:", pre.data_source)
print("scientific_evaluation:", pre.scientific_evaluation)

if not pre.ok:
    result = run_notebook_audit(
        layout=layout,
        data_root=DATA_ROOT,
        output_dir=OUT_DIR,
        sample_size=SAMPLE_SIZE,
        data_source=pre.data_source,
    )
    print("Wrote DATA_NOT_AVAILABLE artefacts to", OUT_DIR)
    raise SystemExit(pre.message)

result = run_notebook_audit(
    layout=layout,
    data_root=DATA_ROOT if DATA_ROOT is not None else pre.data_root,
    output_dir=OUT_DIR,
    sample_size=SAMPLE_SIZE,
    data_source=None if DATA_ROOT is None else os.environ.get("RSNA_AUDIT_DATA_SOURCE"),
)

print("status=", result["status"])
print("DATA_SOURCE=", result["data_source"])
print("scientific_evaluation=", result["scientific_evaluation"])
print("training_run=", result["training_run"], "folds_created=", result["folds_created"])
print("output_files:")
for name, path in result["output_files"].items():
    print(f"  {name}: {path}")
if result["status"] == STATUS_DATA_NOT_AVAILABLE:
    raise SystemExit("AUDIT STOPPED: required inputs missing")


In [ ]:
# Display aggregate tables only. Do not print Report text or pixel arrays.
import pandas as pd

out = Path(result["output_files"]["target_prevalence.csv"]).parent
print("output_dir=", out)
print("DATA_SOURCE=", result["data_source"])
print("provenance Level A:", result["level_a"]["provenance"])
print("provenance Level B:", result["level_b"]["provenance"])
print("provenance Level C:", result["level_c"]["provenance"])
print("is_full_dataset_statistic (Level C):", result["level_c"]["is_full_dataset_statistic"])
print("patient_grouped_folds_created:", result["level_b"].get("patient_grouped_folds_created"))

for name in (
    "target_prevalence.csv",
    "series_summary.csv",
    "dicom_header_summary.csv",
    "dicom_qc_sample.csv",
    "audit_manifest.json",
):
    path = out / name
    print(f"\n=== {name} ===")
    if name.endswith(".json"):
        print(Path(path).read_text(encoding="utf-8")[:4000])
    else:
        frame = pd.read_csv(path)
        assert "Report" not in frame.columns
        print(frame.head(30).to_string(index=False))

print("\nLevel C sample reasons are documented findings, not prevalence:")
for row in result["level_c"].get("sample") or []:
    print(" ", row["sample_reason"], row.get("StudyInstanceUID"))
